# Newcomer retention: event-based vs. organic newcomers

Matched-pair observational study (MCEQ 12576, PUC-Campinas). See `PROTOCOL.md` for the pre-registered analysis plan.

**Unit of analysis:** pair (event-based newcomer, matched organic newcomer from the same project).
**Dependent variable:** activity duration in months (first to last contribution in the project).
**D** = duration(event) − duration(organic).

**Analysis sets** (`PROTOCOL.md`, Amendment 1): the *primary* analysis leaves out the 65 self-matched pairs, where both members are the same GitHub account (n = 1936); the *sensitivity* analysis repeats it on all 2001 pairs.

Sections 1–5 load, validate and describe the data. Sections 6–9 are the confirmatory analysis on the primary set, section 10 the sensitivity analysis and section 11 the exploratory analysis. Every number is saved to `results/results.json`.

Prerequisite: run `uv run python src/prepare_data.py` to generate `data/processed/pairs.csv`.

## 1. Setup

In [1]:
# --- IMPORTS ---
from pathlib import Path
from scipy import stats

import numpy as np
import pandas as pd


# --- GLOBALS ---
# repository root, found from either the root or notebooks/
REPO_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'PROTOCOL.md').exists()
)
PAIRS_FILE = REPO_ROOT / 'data' / 'processed' / 'pairs.csv'

# analysis parameters fixed by PROTOCOL.md
ALPHA = 0.05

# sizes the dataset must have, as reported by the replication package
EXPECTED_PAIRS = 2001
EXPECTED_REPOS = 330
EXPECTED_EVENT_COUNTS = {
    'gsoc': 739,
    '24pr': 576,
    'hacktoberfest': 516,
    'lfx': 170,
}

pd.set_option('display.precision', 2)

## 2. Load data

In [2]:
pairs = pd.read_csv(PAIRS_FILE)

# primary set leaves out self-matched pairs; sensitivity set keeps them all
analysis_sets = {
    'primary': pairs[~pairs['self_matched']],
    'sensitivity': pairs,
}

print(pairs.dtypes)
pairs.head()

pair_id               str
repo                  str
event_type            str
mentorship           bool
duration_event      int64
duration_organic    int64
D                   int64
self_matched         bool
dtype: object


,pair_id,repo,event_type,mentorship,duration_event,duration_organic,D,self_matched
0,0rphee__hacktoberfest__haskell__haskell-langua...,haskell/haskell-language-server,hacktoberfest,False,36,1,35,False
1,0shine0__gsoc__movingblocks__terasology,movingblocks/terasology,gsoc,True,6,3,3,False
2,0xflotus__24pr__24pullrequests__24pullrequests,24pullrequests/24pullrequests,24pr,False,1,4,-3,False
3,0xflotus__24pr__apache__nifi,apache/nifi,24pr,False,1,101,-100,False
4,1000miles__24pr__sozialhelden__wheelmap,sozialhelden/wheelmap,24pr,False,32,1,31,False


## 3. Validations

The same checks enforced by `src/prepare_data.py`, run again on the processed file.

In [3]:
durations = pairs[['duration_event', 'duration_organic']]
counts = pairs['event_type'].value_counts().to_dict()
repos = pairs['repo'].nunique()
mentored = pairs['event_type'].isin(['gsoc', 'lfx'])
difference = pairs['duration_event'] - pairs['duration_organic']

# each check, mapped to whether it passed
checks = {
    f'{EXPECTED_PAIRS} pairs (found {len(pairs)})': (
        len(pairs) == EXPECTED_PAIRS
    ),
    'no duplicated pair ids': not pairs['pair_id'].duplicated().any(),
    'no missing durations': not durations.isna().any(axis=None),
    'durations >= 1 month': (durations >= 1).all(axis=None),
    f'{EXPECTED_REPOS} repos (found {repos})': repos == EXPECTED_REPOS,
    f'pairs per event: {counts}': counts == EXPECTED_EVENT_COUNTS,
    'mentorship flag matches event type': (
        pairs['mentorship'] == mentored
    ).all(),
    'D = duration_event - duration_organic': (pairs['D'] == difference).all(),
    'self_matched is a boolean flag': pairs['self_matched'].dtype == bool,
}

# show every check, then fail listing the ones that did not pass
status = {True: 'ok', False: 'FAIL'}
for check, passed in checks.items():
    print(f'[{status[bool(passed)]}] {check}')

failed = [check for check, passed in checks.items() if not passed]
assert not failed, f'Failed checks: {failed}'

[ok] 2001 pairs (found 2001)
[ok] no duplicated pair ids
[ok] no missing durations
[ok] durations >= 1 month
[ok] 330 repos (found 330)
[ok] pairs per event: {'gsoc': 739, '24pr': 576, 'hacktoberfest': 516, 'lfx': 170}
[ok] mentorship flag matches event type
[ok] D = duration_event - duration_organic
[ok] self_matched is a boolean flag


## 4. Counts

In [4]:
print(
    f'Pairs: {len(pairs)}  |  Contributors: {2 * len(pairs)}  |  '
    f'Repositories: {pairs["repo"].nunique()}'
)

# pairs per event type, mentored events first
counts = (
    pairs.groupby(['mentorship', 'event_type'])
    .agg(pairs=('pair_id', 'size'), self_matched=('self_matched', 'sum'))
    .reset_index()
    .sort_values(['mentorship', 'pairs'], ascending=[False, False])
)
counts['primary'] = counts['pairs'] - counts['self_matched']
counts['primary_%'] = 100 * counts['primary'] / counts['primary'].sum()
counts

Pairs: 2001  |  Contributors: 4002  |  Repositories: 330


,mentorship,event_type,pairs,self_matched,primary,primary_%
2,True,gsoc,739,13,726,37.50
3,True,lfx,170,12,158,8.16
0,False,24pr,576,9,567,29.29
1,False,hacktoberfest,516,31,485,25.05


In [5]:
pairs_per_repo = pairs.groupby('repo').size()
pairs_per_repo.describe().to_frame('pairs per repo').T

,count,mean,std,min,25%,50%,75%,max
pairs per repo,330.0,6.06,5.83,1.0,2.0,4.0,8.0,45.0


## 5. Overall descriptive statistics of activity duration (months), by group

For each analysis set. Descriptive only; no inferential comparison is made here.

In [6]:
def describe_durations(pairs: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize the activity duration of each group.

    :param pairs: The pairs to summarize.

    :return: n, mean, median, sd, min and max per group.
    """
    # one column per group, one row per pair
    durations = pairs[['duration_event', 'duration_organic']].rename(
        columns={'duration_event': 'event', 'duration_organic': 'organic'}
    )

    return durations.agg(['count', 'mean', 'median', 'std', 'min', 'max']).T


# one block of rows per analysis set
descriptive = pd.concat(
    {name: describe_durations(data) for name, data in analysis_sets.items()}
)
descriptive.columns = ['n', 'mean', 'median', 'sd', 'min', 'max']
descriptive.astype({'n': int, 'min': int, 'max': int})

n   mean  median     sd  min  max
primary     event    1936  21.11     8.0  30.06    1  188
            organic  1936  21.17     6.0  31.93    1  272
sensitivity event    2001  21.07     8.0  29.80    1  188
            organic  2001  21.34     7.0  31.91    1  272

---
# Confirmatory analysis

Sections 6–9 use the primary set (n = 1936), following `PROTOCOL.md` sections 2–4.

## 6. Normality of D (Shapiro–Wilk, α = 0.05)

In [7]:
# test the protocol prescribes, by whether normality of D is rejected
TEST_BY_NORMALITY_REJECTED = {
    False: 'paired t-test',
    True: 'Wilcoxon signed-rank',
}

# how each kind of value is printed
FORMATS = {float: '{:.6g}'}


def show(result: dict) -> None:
    """
    Print each entry of a result, one per line.

    :param result: The result to print.

    :return: None.
    """
    for key, value in result.items():
        text = FORMATS.get(type(value), '{}').format(value)
        print(f'  {key:<20} {text}')


def test_normality(d: np.ndarray) -> dict:
    """
    Run the Shapiro-Wilk test on D and pick the main test from its result.

    :param d: The paired differences.

    :return: W, p, whether normality is rejected and the test to use.
    """
    result = stats.shapiro(d)
    rejected = bool(result.pvalue < ALPHA)

    return {
        'W': float(result.statistic),
        'p': float(result.pvalue),
        'normality_rejected': rejected,
        'main_test': TEST_BY_NORMALITY_REJECTED[rejected],
    }


primary_d = analysis_sets['primary']['D'].to_numpy()
normality = test_normality(primary_d)

print(f'Shapiro-Wilk on D, primary set (n = {len(primary_d)}):')
show(normality)

Shapiro-Wilk on D, primary set (n = 1936):
  W                    0.89705
  p                    3.54789e-34
  normality_rejected   True
  main_test            Wilcoxon signed-rank
